# **Day 35: CSV Files**

**CSV** means **Comma Separated Values**. It is a simple text file where each line is a row, and the values are separated by commas. Excel can open it, and almost all data (including GPS data) is shared in this format.

```
district,hq,population
Kolkata,Kolkata,4496694
Howrah,Howrah,4850029
```

Python has a built-in `csv` module. It handles tricky cases correctly, for example when a value itself contains a comma.

## **Writing a CSV File**

In [1]:
import csv

rows = [
    ["district", "hq", "division", "population_2011"],
    ["Kolkata", "Kolkata", "Presidency", 4496694],
    ["Howrah", "Howrah", "Presidency", 4850029],
    ["Paschim Medinipur", "Midnapore", "Medinipur", 5913457],
    ["Purba Medinipur", "Tamluk", "Medinipur", 5095875],
    ["Darjeeling", "Darjeeling", "Jalpaiguri", 1846823],
    ["Jalpaiguri", "Jalpaiguri", "Jalpaiguri", 3872846],
    ["Bankura", "Bankura", "Medinipur", 3596674],
    ["Purulia", "Purulia", "Medinipur", 2930115],
]

with open("districts.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print(open("districts.csv", encoding="utf-8").read())

district,hq,division,population_2011
Kolkata,Kolkata,Presidency,4496694
Howrah,Howrah,Presidency,4850029
Paschim Medinipur,Midnapore,Medinipur,5913457
Purba Medinipur,Tamluk,Medinipur,5095875
Darjeeling,Darjeeling,Jalpaiguri,1846823
Jalpaiguri,Jalpaiguri,Jalpaiguri,3872846
Bankura,Bankura,Medinipur,3596674
Purulia,Purulia,Medinipur,2930115



- `newline=""` is needed on Windows, otherwise empty lines appear between rows.

## **Reading a CSV File**

In [2]:
with open("districts.csv", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)          # The first line is the header
    print(header)
    for row in reader:
        print(row)

['district', 'hq', 'division', 'population_2011']
['Kolkata', 'Kolkata', 'Presidency', '4496694']
['Howrah', 'Howrah', 'Presidency', '4850029']
['Paschim Medinipur', 'Midnapore', 'Medinipur', '5913457']
['Purba Medinipur', 'Tamluk', 'Medinipur', '5095875']
['Darjeeling', 'Darjeeling', 'Jalpaiguri', '1846823']
['Jalpaiguri', 'Jalpaiguri', 'Jalpaiguri', '3872846']
['Bankura', 'Bankura', 'Medinipur', '3596674']
['Purulia', 'Purulia', 'Medinipur', '2930115']


**Note:** Every value is read as a **string**. We must convert numbers ourselves.

## **DictReader: Each Row as a Dictionary**
`DictReader` uses the header as keys. This makes the code much easier to read.

In [3]:
with open("districts.csv", encoding="utf-8") as f:
    data = list(csv.DictReader(f))

print(data[0])

for d in data:
    d["population_2011"] = int(d["population_2011"])

# Total population by division
by_division = {}
for d in data:
    by_division[d["division"]] = by_division.get(d["division"], 0) + d["population_2011"]

for div, pop in sorted(by_division.items(), key=lambda x: -x[1]):
    print(f"{div:<12}{pop:>12,}")

{'district': 'Kolkata', 'hq': 'Kolkata', 'division': 'Presidency', 'population_2011': '4496694'}
Medinipur     17,536,121
Presidency     9,346,723
Jalpaiguri     5,719,669


## **Example from GIS: GPS Field Survey Points**
During a field survey, a GPS device records the location of each point we visit, and we note down what we saw (land cover). The data is usually saved as a CSV file.

Let us create such a file for a survey near **Kharagpur**, then read it and find how many points of each land cover type we collected.

In [4]:
survey = """point_id,latitude,longitude,land_cover,photo
KGP01,22.3302,87.3237,Built-up,IMG_101.jpg
KGP02,22.3190,87.2985,Agriculture,IMG_102.jpg
KGP03,22.3411,87.3102,Forest,IMG_103.jpg
KGP04,22.3120,87.3350,Water,IMG_104.jpg
KGP05,22.3265,87.2890,Agriculture,IMG_105.jpg
KGP06,22.3350,87.3421,Built-up,IMG_106.jpg
KGP07,22.3055,87.3010,Agriculture,IMG_107.jpg
"""
with open("kgp_survey.csv", "w", encoding="utf-8") as f:
    f.write(survey)

with open("kgp_survey.csv", encoding="utf-8") as f:
    points = list(csv.DictReader(f))

counts = {}
for p in points:
    counts[p["land_cover"]] = counts.get(p["land_cover"], 0) + 1

print("Points collected:", len(points))
for cls, n in counts.items():
    print(f"  {cls:<12} {n}")

lats = [float(p["latitude"]) for p in points]
lons = [float(p["longitude"]) for p in points]
print("Survey area -> lat:", min(lats), "to", max(lats), " lon:", min(lons), "to", max(lons))

Points collected: 7
  Built-up     2
  Agriculture  3
  Forest       1
  Water        1
Survey area -> lat: 22.3055 to 22.3411  lon: 87.289 to 87.3421


## **DictWriter: Writing Dictionaries**
Let us save only the agriculture points into a new CSV file.

In [5]:
agri = [p for p in points if p["land_cover"] == "Agriculture"]

with open("agri_points.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["point_id", "latitude", "longitude"])
    writer.writeheader()
    for p in agri:
        writer.writerow({k: p[k] for k in ["point_id", "latitude", "longitude"]})

print(open("agri_points.csv", encoding="utf-8").read())

point_id,latitude,longitude
KGP02,22.3190,87.2985
KGP05,22.3265,87.2890
KGP07,22.3055,87.3010



## **Values That Contain Commas**

In [6]:
with open("tricky.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["name", "address"])
    w.writerow(["Indian Museum", "27, Jawaharlal Nehru Road, Kolkata"])

print(open("tricky.csv", encoding="utf-8").read())
with open("tricky.csv", encoding="utf-8") as f:
    for row in csv.reader(f):
        print(row)

name,address
Indian Museum,"27, Jawaharlal Nehru Road, Kolkata"

['name', 'address']
['Indian Museum', '27, Jawaharlal Nehru Road, Kolkata']


The `csv` module put quotes around the address automatically, and read it back correctly as one value. A simple `split(",")` would have broken it into many pieces.

In [7]:
# Quick question
# Guess the output of this code cell
# import csv, io
# row = next(csv.reader(io.StringIO('a,"b,c",d')))
# print(len(row), row)